In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import math
%matplotlib inline


In [ ]:
# read in all the words
words = open('names.txt', 'r').read().splitlines()
print(len(words))
print(max(len(w) for w in words))
print(words[:8])


In [ ]:
# build the vocabulary of characters and mappings to/from integers
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
vocab_size = len(itos)
print(itos)
print(vocab_size)


In [ ]:
# shuffle up the words
import random
random.seed(42)
random.shuffle(words)


In [ ]:
# build the dataset
block_size = 8 # context length: how many characters do we take to predict the next one?

def build_dataset(words):  
  X, Y = [], []
  
  for w in words:
    context = [0] * block_size
    for ch in w + '.':
      ix = stoi[ch]
      X.append(context)
      Y.append(ix)
      context = context[1:] + [ix] # crop and append

  X = torch.tensor(X)
  Y = torch.tensor(Y)
  print(X.shape, Y.shape)
  return X, Y

n1 = int(0.8*len(words))
n2 = int(0.9*len(words))
Xtr,  Ytr  = build_dataset(words[:n1])     # 80%
Xdev, Ydev = build_dataset(words[n1:n2])   # 10%
Xte,  Yte  = build_dataset(words[n2:])     # 10%


In [ ]:
for x,y in zip(Xtr[:20], Ytr[:20]):
  print(''.join(itos[ix.item()] for ix in x), '-->', itos[y.item()])


In [ ]:
# -----------------------------------------------------------------------------------------------
class LayerNorm:
    """ LayerNorm but with an optional bias. PyTorch doesn't support simply bias=False """

    def __init__(self, ndim, bias):
        self.weight = torch.ones(ndim)
        self.bias = torch.zeros(ndim) if bias else None

    def __call__(self, input):
        return F.layer_norm(input, self.weight.shape, self.weight, self.bias, 1e-5)
    
    def parameters(self):
        return [self.weight] + ([] if self.bias is None else [self.bias])

# -----------------------------------------------------------------------------------------------
class CausalSelfAttention:
    """
    A vanilla multi-head masked self-attention layer with a projection at the end.
    """

    def __init__(self, n_embd, n_head, block_size, bias=True):
        assert n_embd % n_head == 0
        # key, query, value projections for all heads, but in a batch
        self.c_attn_weight = torch.randn(n_embd, 3 * n_embd) / math.sqrt(n_embd)
        self.c_attn_bias = torch.zeros(3 * n_embd) if bias else None
        # output projection
        self.c_proj_weight = torch.randn(n_embd, n_embd) / math.sqrt(n_embd)
        self.c_proj_bias = torch.zeros(n_embd) if bias else None
        # regularization
        self.n_head = n_head
        self.n_embd = n_embd
        # causal mask to ensure that attention is only applied to the left in the input sequence
        self.register_buffer("bias", torch.tril(torch.ones(block_size, block_size))
                                     .view(1, 1, block_size, block_size))

    def register_buffer(self, name, tensor):
        # Simple way to store the causal mask
        setattr(self, name, tensor)

    def __call__(self, x):
        B, T, C = x.size() # batch size, sequence length, embedding dimensionality (n_embd)

        # calculate query, key, values for all heads in batch and move head forward to be the batch dim
        q, k, v  = (x @ self.c_attn_weight + (self.c_attn_bias if self.c_attn_bias is not None else 0)).split(self.n_embd, dim=2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)

        # causal self-attention; Self-attend: (B, nh, T, hs) x (B, nh, hs, T) -> (B, nh, T, T)
        att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))
        att = att.masked_fill(self.bias[:,:,:T,:T] == 0, float('-inf'))
        att = F.softmax(att, dim=-1)
        y = att @ v # (B, nh, T, T) x (B, nh, T, hs) -> (B, nh, T, hs)
        y = y.transpose(1, 2).contiguous().view(B, T, C) # re-assemble all head outputs side by side

        # output projection
        y = y @ self.c_proj_weight + (self.c_proj_bias if self.c_proj_bias is not None else 0)
        self.out = y
        return y
    
    def parameters(self):
        params = [self.c_attn_weight, self.c_proj_weight]
        if self.c_attn_bias is not None:
            params.append(self.c_attn_bias)
        if self.c_proj_bias is not None:
            params.append(self.c_proj_bias)
        return params


In [ ]:
# -----------------------------------------------------------------------------------------------
class MLP:
    """
    Simple feed-forward network
    """

    def __init__(self, n_embd, bias=True):
        self.c_fc_weight = torch.randn(n_embd, 4 * n_embd) / math.sqrt(n_embd)
        self.c_fc_bias = torch.zeros(4 * n_embd) if bias else None
        self.c_proj_weight = torch.randn(4 * n_embd, n_embd) / math.sqrt(4 * n_embd)
        self.c_proj_bias = torch.zeros(n_embd) if bias else None

    def __call__(self, x):
        x = x @ self.c_fc_weight + (self.c_fc_bias if self.c_fc_bias is not None else 0)
        x = F.gelu(x)
        x = x @ self.c_proj_weight + (self.c_proj_bias if self.c_proj_bias is not None else 0)
        self.out = x
        return x
    
    def parameters(self):
        params = [self.c_fc_weight, self.c_proj_weight]
        if self.c_fc_bias is not None:
            params.append(self.c_fc_bias)
        if self.c_proj_bias is not None:
            params.append(self.c_proj_bias)
        return params

# -----------------------------------------------------------------------------------------------
class Block:
    """
    Transformer block: communication followed by computation
    """

    def __init__(self, n_embd, n_head, block_size, bias=True):
        self.ln_1 = LayerNorm(n_embd, bias)
        self.attn = CausalSelfAttention(n_embd, n_head, block_size, bias)
        self.ln_2 = LayerNorm(n_embd, bias)
        self.mlp = MLP(n_embd, bias)

    def __call__(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        self.out = x
        return x
    
    def parameters(self):
        params = []
        params.extend(self.ln_1.parameters())
        params.extend(self.attn.parameters())
        params.extend(self.ln_2.parameters())
        params.extend(self.mlp.parameters())
        return params


In [ ]:
# -----------------------------------------------------------------------------------------------
class Embedding:
  
  def __init__(self, num_embeddings, embedding_dim):
    self.weight = torch.randn((num_embeddings, embedding_dim))
    
  def __call__(self, IX):
    self.out = self.weight[IX]
    return self.out
  
  def parameters(self):
    return [self.weight]

# -----------------------------------------------------------------------------------------------
class GPT:
    """
    GPT Language Model
    """

    def __init__(self, vocab_size, n_embd, n_head, n_layer, block_size, bias=True):
        # token embedding
        self.wte = Embedding(vocab_size, n_embd)
        # position embedding
        self.wpe = Embedding(block_size, n_embd)
        # transformer blocks
        self.h = [Block(n_embd, n_head, block_size, bias) for _ in range(n_layer)]
        # final layer norm
        self.ln_f = LayerNorm(n_embd, bias)
        # language modeling head
        self.lm_head_weight = torch.randn(n_embd, vocab_size) / math.sqrt(n_embd)
        
        self.block_size = block_size

    def __call__(self, idx):
        device = idx.device
        b, t = idx.size()
        assert t <= self.block_size, f"Cannot forward sequence of length {t}, block size is only {self.block_size}"
        pos = torch.arange(0, t, dtype=torch.long, device=device) # shape (t)

        # forward the GPT model itself
        tok_emb = self.wte(idx) # token embeddings of shape (b, t, n_embd)
        pos_emb = self.wpe(pos) # position embeddings of shape (t, n_embd)
        x = tok_emb + pos_emb # (b, t, n_embd)
        
        for block in self.h:
            x = block(x)
        x = self.ln_f(x)

        logits = x @ self.lm_head_weight # (b, t, vocab_size)
        self.out = logits
        return logits
    
    def parameters(self):
        params = []
        params.extend(self.wte.parameters())
        params.extend(self.wpe.parameters())
        for block in self.h:
            params.extend(block.parameters())
        params.extend(self.ln_f.parameters())
        params.append(self.lm_head_weight)
        return params


In [ ]:
torch.manual_seed(42)


In [ ]:
# create the model
model = GPT(
    vocab_size=vocab_size,
    n_embd=64,          # embedding dimension
    n_head=4,           # number of attention heads
    n_layer=4,          # number of transformer blocks
    block_size=block_size,
    bias=True
)

parameters = model.parameters()
print(f"number of parameters: {sum(p.nelement() for p in parameters)}")
for p in parameters:
    p.requires_grad = True


In [ ]:
# Training loop
max_steps = 50000
batch_size = 64
lossi = []

for i in range(max_steps):
  
  # minibatch construct
  ix = torch.randint(0, Xtr.shape[0], (batch_size,))
  Xb, Yb = Xtr[ix], Ytr[ix] # batch X,Y
  
  # forward pass
  logits = model(Xb)
  # we only care about the last time step for next character prediction
  logits = logits[:, -1, :] # (B, vocab_size)
  loss = F.cross_entropy(logits, Yb) # loss function
  
  # backward pass
  for p in parameters:
    p.grad = None
  loss.backward()
  
  # update: simple SGD
  lr = 0.1 if i < 30000 else 0.01 # step learning rate decay
  for p in parameters:
    p.data += -lr * p.grad

  # track stats
  if i % 5000 == 0: # print every once in a while
    print(f'{i:7d}/{max_steps:7d}: {loss.item():.4f}')
  lossi.append(loss.log10().item())


In [ ]:
plt.plot(torch.tensor(lossi).view(-1, 1000).mean(1))
plt.title('Training Loss')
plt.xlabel('Steps (x1000)')
plt.ylabel('Loss (log10)')


In [ ]:
# evaluate the loss
@torch.no_grad() # this decorator disables gradient tracking inside pytorch
def split_loss(split):
  x,y = {
    'train': (Xtr, Ytr),
    'val': (Xdev, Ydev),
    'test': (Xte, Yte),
  }[split]
  logits = model(x)
  logits = logits[:, -1, :] # only the last time step
  loss = F.cross_entropy(logits, y)
  print(split, loss.item())

split_loss('train')
split_loss('val')


In [ ]:
# sample from the model
@torch.no_grad()
def generate(model, max_new_tokens=50, temperature=1.0):
    # start with just the starting token
    context = torch.zeros((1, 1), dtype=torch.long)  # start with '.'
    
    for _ in range(max_new_tokens):
        # if the sequence context is growing too long we must crop it at block_size
        context_cond = context if context.size(1) <= block_size else context[:, -block_size:]
        # forward the model to get the logits for the next token
        logits = model(context_cond)
        # pluck the logits at the final step and scale by desired temperature
        logits = logits[:, -1, :] / temperature
        # apply softmax to convert logits to (normalized) probabilities
        probs = F.softmax(logits, dim=-1)
        # sample from the distribution
        next_token = torch.multinomial(probs, num_samples=1)
        # append sampled index to the running sequence and continue
        context = torch.cat((context, next_token), dim=1)
        # if we hit the end token, break
        if next_token.item() == 0:  # '.' is our end token
            break
    
    return context[0].tolist()

# Generate some names
for _ in range(20):
    out = generate(model, max_new_tokens=20)
    name = ''.join(itos[i] for i in out[1:])  # skip the initial '.'
    print(name)


In [ ]:
# Let's look at the attention patterns
# We'll take a single example and see what each head is attending to

def visualize_attention(model, text, head_idx=0, layer_idx=0):
    """Visualize attention patterns for a given text"""
    # Convert text to indices
    context = [0] * (len(text) + 1)  # pad with starting token
    for i, ch in enumerate(text):
        context[i+1] = stoi[ch]
    
    x = torch.tensor([context])
    
    # Forward pass to get attention weights
    with torch.no_grad():
        # Get embeddings
        pos = torch.arange(0, x.size(1))
        tok_emb = model.wte(x)
        pos_emb = model.wpe(pos)
        x_emb = tok_emb + pos_emb
        
        # Pass through layers up to the target layer
        for i in range(layer_idx + 1):
            if i == layer_idx:
                # For the target layer, we'll intercept the attention weights
                x_norm = model.h[i].ln_1(x_emb)
                
                # Get attention weights manually
                attn = model.h[i].attn
                B, T, C = x_norm.size()
                
                q, k, v = (x_norm @ attn.c_attn_weight + (attn.c_attn_bias if attn.c_attn_bias is not None else 0)).split(attn.n_embd, dim=2)
                k = k.view(B, T, attn.n_head, C // attn.n_head).transpose(1, 2)
                q = q.view(B, T, attn.n_head, C // attn.n_head).transpose(1, 2)
                
                att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))
                att = att.masked_fill(attn.bias[:,:,:T,:T] == 0, float('-inf'))
                att_weights = F.softmax(att, dim=-1)
                
                # Extract the specific head
                head_att = att_weights[0, head_idx, :, :].cpu().numpy()
                
                break
            else:
                x_emb = model.h[i](x_emb)
    
    return head_att, context

# Visualize attention for a sample name
sample_name = "emma"
att_weights, context = visualize_attention(model, sample_name, head_idx=0, layer_idx=0)

# Create visualization
plt.figure(figsize=(8, 6))
labels = ['.'] + list(sample_name) + ['.']

plt.imshow(att_weights[:len(labels), :len(labels)], cmap='Blues')
plt.colorbar()
plt.xticks(range(len(labels)), labels)
plt.yticks(range(len(labels)), labels)
plt.xlabel('Keys (what we attend to)')
plt.ylabel('Queries (what position is attending)')
plt.title(f'Attention Pattern for "{sample_name}" (Head 0, Layer 0)')

# Add text annotations
for i in range(len(labels)):
    for j in range(len(labels)):
        if j <= i:  # only show lower triangular (causal)
            plt.text(j, i, f'{att_weights[i, j]:.2f}', 
                    ha='center', va='center', 
                    color='white' if att_weights[i, j] > 0.5 else 'black')

plt.tight_layout()
plt.show()


In [ ]:
# Let's compare different attention heads
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

sample_name = "sophia"
for head_idx in range(4):  # assuming 4 heads
    att_weights, context = visualize_attention(model, sample_name, head_idx=head_idx, layer_idx=0)
    labels = ['.'] + list(sample_name) + ['.']
    
    im = axes[head_idx].imshow(att_weights[:len(labels), :len(labels)], cmap='Blues')
    axes[head_idx].set_xticks(range(len(labels)))
    axes[head_idx].set_xticklabels(labels)
    axes[head_idx].set_yticks(range(len(labels)))
    axes[head_idx].set_yticklabels(labels)
    axes[head_idx].set_title(f'Head {head_idx}')
    
    if head_idx >= 2:
        axes[head_idx].set_xlabel('Keys')
    if head_idx % 2 == 0:
        axes[head_idx].set_ylabel('Queries')

plt.suptitle(f'Different Attention Heads for "{sample_name}" (Layer 0)')
plt.tight_layout()
plt.show()
